# DEL ACERO AL ALGORITMO
## ¿Desde qué base de talento parte el Biobío que quiere avanzar hacia la Industria 4.0?

**Trabajo Práctico — Estadística Descriptiva**  
**Base de datos:** Matrículas de Educación Superior de la Región del Biobío, 2021.

### Pregunta central
**En la fotografía educativa de 2021, ¿qué peso tenía la formación ligada a los motores productivos del Biobío frente a las capacidades que pueden transformarlos mediante digitalización, automatización y sustentabilidad?**


## Objetivos

1. Describir estadísticamente la base de matrículas de educación superior del Biobío.
2. Aplicar tablas de frecuencia, gráficos, medidas de tendencia central, percentiles y medidas de dispersión.
3. Responder las tres preguntas obligatorias del trabajo.
4. Construir una aplicación regional llamada **“Del acero al algoritmo”**, comparando carreras vinculadas a motores productivos con carreras asociadas a capacidades transformadoras.

> **Importante:** el análisis es descriptivo. La base permite observar patrones, pero no demuestra causalidad ni déficit de trabajadores.


## Contexto regional

El Biobío posee una tradición productiva ligada a industria, manufactura, construcción, forestal, pesca, agro y logística. Al mismo tiempo, planes regionales recientes buscan fortalecer manufactura avanzada, tecnologías digitales e Industria 4.0.

En este trabajo usamos la matrícula de **2021** como una fotografía del punto de partida educativo. Los planes regionales posteriores sirven solamente como contexto.

**Fuentes de contexto regional:**
- Gobierno Regional del Biobío, Centro Tecnológico de Manufactura Avanzada e Industria 4.0.
- CORFO, iniciativas de manufactura avanzada e Industria 4.0 en Biobío.


# 1. Población, muestra y carga de datos

### ¿Cuál es la población del estudio?
La población de interés corresponde a las matrículas de educación superior de la Región del Biobío durante 2021.

### ¿Cuál es la muestra o base disponible?
La base entregada contiene los registros que utilizaremos para realizar el estudio. Primero calcularemos cuántas filas contiene.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#Creación del DataFrame con la base de datos
df = pd.read_excel('11_MATRICULAS_ED_SUPERIOR_BIOBIO_2021.xlsx',
                   sheet_name='BASE DE DATOS')


In [ ]:
#Número de registros de la base original
muestra = df.shape[0]
muestra


La base original contiene **106.555 registros de matrícula**.


### Primeras cinco filas del DataFrame


In [ ]:
#Primeras 5 filas del DataFrame
df.head(5)


### Últimas cinco filas del DataFrame


In [ ]:
#Últimas 5 filas del DataFrame
df.tail(5)


## Clasificación de las variables principales

| Variable | Tipo de variable |
|---|---|
| `GENERO` | Cualitativa nominal |
| `EDAD` | Cuantitativa discreta |
| `RANGO EDAD` | Cualitativa ordinal |
| `TIPO DE INSTITUCION` | Cualitativa nominal |
| `AREA CONOCIMIENTO` | Cualitativa nominal |
| `JORNADA` | Cualitativa nominal |
| `DURACION TOTAL CARRERA (SEMESTRES)` | Cuantitativa discreta |
| `VALOR ARANCEL (PESOS)` | Cuantitativa continua |
| `PROVINCIA SEDE` | Cualitativa nominal |


# 2. Preparación de la base de trabajo

Para comparar carreras de un mismo nivel se consideran solamente **Carreras Profesionales** y **Carreras Técnicas**.

Además, para los análisis de arancel se excluyen registros cuyo arancel sea igual a $0, porque ese valor no permite realizar una comparación de precios válida.


In [ ]:
#Filtramos solamente carreras de pregrado
pregrado = ['Carreras Profesionales', 'Carreras Tecnicas']
d = df[df['NIVEL CARRERA'].isin(pregrado)].copy()

#Eliminamos registros con arancel igual a $0
d = d[d['VALOR ARANCEL (PESOS)'] > 0].copy()

#Tamaño de la base de trabajo
n = d.shape[0]
n


Después de aplicar estos criterios, la base de trabajo contiene **101.067 matrículas**.


## Oferta académica única

Para analizar precios no debemos contar el mismo arancel una vez por cada estudiante matriculado.

Por eso creamos una segunda base llamada `ofertas`, donde una oferta se identifica por la combinación de institución, carrera, comuna, modalidad, jornada, duración, matrícula y arancel.


In [ ]:
#Columnas utilizadas para identificar una oferta académica
clave_oferta = [
    'NOMBRE DE INSTITUCION',
    'NOMBRE CARRERA',
    'COMUNA SEDE',
    'MODALIDAD',
    'JORNADA',
    'DURACION TOTAL CARRERA (SEMESTRES)',
    'VALOR MATRICULA (PESOS)',
    'VALOR ARANCEL (PESOS)'
]

#Eliminamos ofertas repetidas
ofertas = d.drop_duplicates(subset=clave_oferta).copy()

#Cantidad de ofertas académicas únicas
ofertas.shape[0]


Se obtienen **1.273 ofertas académicas únicas**.  
Usaremos `d` cuando la pregunta se refiera a estudiantes/matrículas y `ofertas` cuando la pregunta se refiera al precio de una carrera u oferta.


# 3. ÍTEM 1 — Descripción general de la base


## 3.1 Tabla de frecuencias: área del conocimiento


In [ ]:
#Frecuencia absoluta
cuenta_area = d.groupby('AREA CONOCIMIENTO').size()

#Tabla de frecuencias
tabla_area = pd.DataFrame({
    'Frecuencia absoluta': cuenta_area,
    'Frecuencia absoluta acumulada': cuenta_area.cumsum(),
    'Frecuencia relativa (%)': cuenta_area/n * 100,
    'Frecuencia relativa acumulada (%)': (cuenta_area/n * 100).cumsum()
})

#Ordenamos de mayor a menor frecuencia
tabla_area = tabla_area.sort_values('Frecuencia absoluta', ascending=False)

round(tabla_area, 2)


**Interpretación:** Tecnología y Salud son las áreas con mayor presencia en la base. Tecnología representa aproximadamente **27,57%** de las matrículas y Salud **24,34%**. Entre ambas concentran más de la mitad de los registros analizados.


In [ ]:
#Variables
categorias_area = tabla_area.index
f_area = tabla_area['Frecuencia absoluta']

#Gráfico
fig, ax = plt.subplots(figsize=(10,5))
barras_area = ax.bar(categorias_area, f_area, edgecolor='black')

#Personalización
ax.set_title('Distribución de matrículas por área del conocimiento')
ax.set_xlabel('Área del conocimiento')
ax.set_ylabel('Cantidad de matrículas')
plt.xticks(rotation=45, ha='right')

plt.show()


## 3.2 Tabla de frecuencias: edad agrupada


In [ ]:
#Creamos intervalos para la edad
d['Intervalos_edad'] = pd.cut(
    d['EDAD'],
    bins=8,
    include_lowest=True,
    precision=0
)

#Frecuencia absoluta
cuenta_edad = d.groupby('Intervalos_edad', observed=True).size()

#Tabla de frecuencias
tabla_edad = pd.DataFrame({
    'Frecuencia absoluta': cuenta_edad,
    'Frecuencia absoluta acumulada': cuenta_edad.cumsum(),
    'Frecuencia relativa (%)': cuenta_edad/n * 100,
    'Frecuencia relativa acumulada (%)': (cuenta_edad/n * 100).cumsum()
})

round(tabla_edad, 2)


In [ ]:
#Variables
categorias_edad = tabla_edad.index.astype(str)
h_edad = tabla_edad['Frecuencia relativa (%)']

#Gráfico
fig, ax = plt.subplots(figsize=(8,5))
barras_edad = ax.bar(categorias_edad, h_edad, edgecolor='black', width=1)

#Personalización
ax.set_title('Distribución de matrículas por edad')
ax.set_xlabel('Edad (años)')
ax.set_ylabel('Frecuencia relativa (%)')
ax.bar_label(barras_edad, fmt='%.1f%%')
plt.xticks(rotation=45, ha='right')

plt.show()


**Interpretación:** la matrícula se concentra principalmente en edades jóvenes. Los intervalos permiten observar cómo disminuye la frecuencia a medida que aumenta la edad.


## 3.3 Tabla de frecuencias: arancel agrupado


In [ ]:
#Creamos intervalos para el arancel
d['Intervalos_arancel'] = pd.cut(
    d['VALOR ARANCEL (PESOS)'],
    bins=8,
    include_lowest=True,
    precision=0
)

#Frecuencia absoluta
cuenta_arancel = d.groupby('Intervalos_arancel', observed=True).size()

#Tabla de frecuencias
tabla_arancel = pd.DataFrame({
    'Frecuencia absoluta': cuenta_arancel,
    'Frecuencia absoluta acumulada': cuenta_arancel.cumsum(),
    'Frecuencia relativa (%)': cuenta_arancel/n * 100,
    'Frecuencia relativa acumulada (%)': (cuenta_arancel/n * 100).cumsum()
})

round(tabla_arancel, 2)


In [ ]:
#Variables
categorias_arancel = tabla_arancel.index.astype(str)
h_arancel = tabla_arancel['Frecuencia relativa (%)']

#Gráfico
fig, ax = plt.subplots(figsize=(9,5))
barras_arancel = ax.bar(categorias_arancel, h_arancel, edgecolor='black', width=1)

#Personalización
ax.set_title('Distribución de matrículas según arancel')
ax.set_xlabel('Intervalos de arancel')
ax.set_ylabel('Frecuencia relativa (%)')
ax.bar_label(barras_arancel, fmt='%.1f%%')
plt.xticks(rotation=45, ha='right')

plt.show()


## 3.4 Medidas de tendencia central


In [ ]:
#Media, mediana y moda del arancel
media_arancel = d['VALOR ARANCEL (PESOS)'].mean()
mediana_arancel = d['VALOR ARANCEL (PESOS)'].median()
moda_arancel = d['VALOR ARANCEL (PESOS)'].mode()[0]

#Media, mediana y moda de la edad
media_edad = d['EDAD'].mean()
mediana_edad = d['EDAD'].median()
moda_edad = d['EDAD'].mode()[0]

#Media, mediana y moda de la duración
media_duracion = d['DURACION TOTAL CARRERA (SEMESTRES)'].mean()
mediana_duracion = d['DURACION TOTAL CARRERA (SEMESTRES)'].median()
moda_duracion = d['DURACION TOTAL CARRERA (SEMESTRES)'].mode()[0]

tabla_tendencia = pd.DataFrame({
    'Media': [media_arancel, media_edad, media_duracion],
    'Mediana': [mediana_arancel, mediana_edad, mediana_duracion],
    'Moda': [moda_arancel, moda_edad, moda_duracion]
}, index=['Arancel', 'Edad', 'Duración (semestres)'])

round(tabla_tendencia, 2)


**Interpretación:** el arancel medio es aproximadamente **$3,22 millones** y la mediana **$3,13 millones**. La cercanía entre ambas indica que el centro de la distribución está alrededor de esos valores, aunque existen aranceles altos que aumentan la media.

La edad mediana es **23 años**, por lo que al menos la mitad de las matrículas corresponde a estudiantes de 23 años o menos.


## 3.5 Percentiles


In [ ]:
#Percentiles del arancel
percentiles_arancel = d['VALOR ARANCEL (PESOS)'].quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

#Percentiles de la edad
percentiles_edad = d['EDAD'].quantile(
    [0.10, 0.25, 0.50, 0.75, 0.90, 0.95]
)

tabla_percentiles = pd.DataFrame({
    'Arancel ($)': percentiles_arancel,
    'Edad': percentiles_edad
})

tabla_percentiles


**Interpretación:** el percentil 50 coincide con la mediana. En arancel, el 50% de las matrículas se encuentra en valores iguales o inferiores a **$3.133.750**. En edad, el 50% se encuentra en **23 años o menos**.


## 3.6 Medidas de dispersión


In [ ]:
#Arancel
rango_arancel = d['VALOR ARANCEL (PESOS)'].max() - d['VALOR ARANCEL (PESOS)'].min()
varianza_arancel = d['VALOR ARANCEL (PESOS)'].var()
desviacion_arancel = d['VALOR ARANCEL (PESOS)'].std()
cv_arancel = desviacion_arancel / d['VALOR ARANCEL (PESOS)'].mean() * 100
ric_arancel = (
    d['VALOR ARANCEL (PESOS)'].quantile(0.75)
    - d['VALOR ARANCEL (PESOS)'].quantile(0.25)
)

#Edad
rango_edad = d['EDAD'].max() - d['EDAD'].min()
varianza_edad = d['EDAD'].var()
desviacion_edad = d['EDAD'].std()
cv_edad = desviacion_edad / d['EDAD'].mean() * 100
ric_edad = d['EDAD'].quantile(0.75) - d['EDAD'].quantile(0.25)

#Duración
rango_duracion = (
    d['DURACION TOTAL CARRERA (SEMESTRES)'].max()
    - d['DURACION TOTAL CARRERA (SEMESTRES)'].min()
)
varianza_duracion = d['DURACION TOTAL CARRERA (SEMESTRES)'].var()
desviacion_duracion = d['DURACION TOTAL CARRERA (SEMESTRES)'].std()
cv_duracion = (
    desviacion_duracion
    / d['DURACION TOTAL CARRERA (SEMESTRES)'].mean()
    * 100
)
ric_duracion = (
    d['DURACION TOTAL CARRERA (SEMESTRES)'].quantile(0.75)
    - d['DURACION TOTAL CARRERA (SEMESTRES)'].quantile(0.25)
)

#Tabla resumen
tabla_dispersion = pd.DataFrame({
    'Arancel': [
        rango_arancel, varianza_arancel, desviacion_arancel,
        cv_arancel, ric_arancel
    ],
    'Edad': [
        rango_edad, varianza_edad, desviacion_edad,
        cv_edad, ric_edad
    ],
    'Duración': [
        rango_duracion, varianza_duracion, desviacion_duracion,
        cv_duracion, ric_duracion
    ]
}, index=[
    'Rango',
    'Varianza',
    'Desviación estándar',
    'CV (%)',
    'RIC'
])

round(tabla_dispersion, 2)


**Interpretación:** el arancel presenta un **coeficiente de variación cercano a 47,1%**, mayor que el de edad y duración. Por lo tanto, proporcionalmente, el arancel es la variable más heterogénea de las tres analizadas.


## 3.7 Otros resúmenes


In [ ]:
#Cantidad de categorías distintas presentes en la base de trabajo
resumen = pd.DataFrame({
    'Cantidad': [
        d['NOMBRE CARRERA'].nunique(),
        d['NOMBRE DE INSTITUCION'].nunique(),
        d['COMUNA SEDE'].nunique(),
        d['PROVINCIA SEDE'].nunique()
    ]
}, index=[
    'Carreras distintas',
    'Instituciones distintas',
    'Comunas distintas',
    'Provincias'
])

resumen


# 4. Aplicación regional — DEL ACERO AL ALGORITMO

Ahora usamos los nombres de las carreras para construir una clasificación propia.

### Motores productivos, “acero”
- Industria y manufactura
- Construcción e infraestructura
- Logística y puertos
- Forestal y madera
- Pesca y acuicultura
- Agroalimentario

### Capacidades transformadoras, “algoritmo”
- Digital y TIC
- Automatización y robótica
- Energía, sustentabilidad y biotecnología

Las carreras que no poseen una relación clara con estos grupos quedan como **Otros campos**. La clasificación es una herramienta creada para este trabajo y no una clasificación oficial.


## 4.1 Clasificación de las carreras

Para mantener el procedimiento visible, primero dejamos todas las carreras como `Otros campos` y luego usamos palabras presentes en `NOMBRE CARRERA` para asignarlas a cada familia.


In [ ]:
#Trabajaremos con el nombre de la carrera en mayúsculas
nombre = d['NOMBRE CARRERA'].str.upper()

#Primero dejamos todas las carreras como otros campos
d['ADN_BIOBIO'] = 'Otros campos'

#Industria y manufactura
d.loc[nombre.str.contains(
    'INGENIERIA CIVIL INDUSTRIAL|INGENIERIA INDUSTRIAL|EJECUCION INDUSTRIAL|'
    'GESTION INDUSTRIAL|MECANICA|ELECTROMECANICA|MANTENIMIENTO INDUSTRIAL|'
    'MANTENCION INDUSTRIAL|ELECTRICIDAD|METALURG|FABRICACION Y MONTAJE INDUSTRIAL|'
    'REFRIGERACION INDUSTRIAL|QUIMICA INDUSTRIAL|QUIMICO ANALISTA INDUSTRIAL|'
    'INGENIERIA CIVIL QUIMICA|MAQUINARIA|DISENO INDUSTRIAL|'
    'INGENIERIA CIVIL DE MATERIALES',
    na=False
), 'ADN_BIOBIO'] = 'Industria y manufactura'

#Energía, sustentabilidad y biotecnología
d.loc[nombre.str.contains(
    'ENERGIA Y SUSTENTABILIDAD|ENERGIAS RENOVABLES|EFICIENCIA ENERGETICA|'
    'INGENIERIA AMBIENTAL|MEDIOAMBIENTE|MEDIO AMBIENTE|'
    'GESTION Y CONTROL AMBIENTAL|QUIMICA AMBIENTAL|BIOTECNOLOGIA VEGETAL',
    na=False
), 'ADN_BIOBIO'] = 'Energia, sustentabilidad y biotecnologia'

#Automatización y robótica
condicion_auto = nombre.str.contains(
    'AUTOMATIZACION|ROBOTICA|MECATRONICA|ELECTRONICA|'
    'INSTRUMENTACION INDUSTRIAL|CONTROL INDUSTRIAL|INSTRUMENTACION Y CONTROL',
    na=False
)

condicion_quirurgica = nombre.str.contains(
    'INSTRUMENTACION QUIRURGICA',
    na=False
)

d.loc[condicion_auto & ~condicion_quirurgica,
      'ADN_BIOBIO'] = 'Automatizacion y robotica'

#Digital y TIC
d.loc[nombre.str.contains(
    'INFORMATICA|COMPUTACION E INFORMATICA|ANALISTA PROGRAMADOR|'
    'PROGRAMACION Y ANALISIS DE SISTEMAS|PROGRAMACION COMPUTACIONAL|'
    'TELECOMUNICACIONES|CONECTIVIDAD Y REDES|CIBERSEGURIDAD|GEOMATICA',
    na=False
), 'ADN_BIOBIO'] = 'Digital y TIC'

#Construcción e infraestructura
condicion_construccion = (
    nombre.str.contains(
        'CONSTRUCCION|CONSTRUCCIONES CIVILES|ARQUITECTURA|MODELAMIENTO ARQUITECTONICO',
        na=False
    )
    | nombre.isin(['INGENIERIA CIVIL', 'INGENIERIA CIVIL-PLAN COMUN'])
)

d.loc[condicion_construccion,
      'ADN_BIOBIO'] = 'Construccion e infraestructura'

#Logística y puertos
d.loc[nombre.str.contains(
    'LOGIST|COMERCIO EXTERIOR|TRANSPORTE MARITIMO|PORTUARIA',
    na=False
), 'ADN_BIOBIO'] = 'Logistica y puertos'

#Agroalimentario
d.loc[nombre.str.contains(
    'AGRONOM|AGROPECUAR|AGRICOLA|GANADERO|PRODUCCION PECUARIA|'
    'GESTION AGROINDUSTRIAL|EN ALIMENTOS',
    na=False
), 'ADN_BIOBIO'] = 'Agroalimentario'

#Forestal y madera
d.loc[nombre.str.contains(
    'FORESTAL|INDUSTRIAS DE LA MADERA',
    na=False
), 'ADN_BIOBIO'] = 'Forestal y madera'

#Pesca y acuicultura
d.loc[nombre.str.contains(
    'ACUICULT|PESQUER',
    na=False
), 'ADN_BIOBIO'] = 'Pesca y acuicultura'


In [ ]:
#Creamos las dos grandes categorías del análisis
motores = [
    'Industria y manufactura',
    'Construccion e infraestructura',
    'Logistica y puertos',
    'Forestal y madera',
    'Pesca y acuicultura',
    'Agroalimentario'
]

transformadoras = [
    'Digital y TIC',
    'Automatizacion y robotica',
    'Energia, sustentabilidad y biotecnologia'
]

#Nueva columna con el bloque general
d['BLOQUE_ADN'] = 'Otros campos'

d.loc[d['ADN_BIOBIO'].isin(motores),
      'BLOQUE_ADN'] = 'Motores productivos'

d.loc[d['ADN_BIOBIO'].isin(transformadoras),
      'BLOQUE_ADN'] = 'Capacidades transformadoras'


## 4.2 Tabla de frecuencias del ADN profesional


In [ ]:
#Frecuencia absoluta
cuenta_bloque = d.groupby('BLOQUE_ADN').size()

#Tabla de frecuencias
tabla_bloque = pd.DataFrame({
    'Frecuencia absoluta': cuenta_bloque,
    'Frecuencia relativa (%)': cuenta_bloque/n * 100
})

round(tabla_bloque, 2)


El resultado principal es:

- **Motores productivos:** 20.350 matrículas, **20,14%**.
- **Capacidades transformadoras:** 7.000 matrículas, **6,93%**.
- **Otros campos:** 73.717 matrículas, **72,94%**.

Para comparar “acero” y “algoritmo” usamos solamente los dos primeros bloques.


In [ ]:
#Seleccionamos los dos bloques principales
tabla_acero_algoritmo = tabla_bloque.loc[
    ['Motores productivos', 'Capacidades transformadoras']
]

#Variables
categorias_adn = tabla_acero_algoritmo.index
h_adn = tabla_acero_algoritmo['Frecuencia relativa (%)']

#Gráfico
fig, ax = plt.subplots(figsize=(7,5))
barras_adn = ax.bar(categorias_adn, h_adn, edgecolor='black')

#Personalización
ax.set_title('Del acero al algoritmo')
ax.set_ylabel('Porcentaje de matrículas')
ax.bar_label(barras_adn, fmt='%.1f%%')

plt.show()


In [ ]:
#Calculamos cuántas veces es mayor el bloque productivo
razon = (
    tabla_acero_algoritmo.loc['Motores productivos', 'Frecuencia absoluta']
    / tabla_acero_algoritmo.loc['Capacidades transformadoras', 'Frecuencia absoluta']
)

round(razon, 1)


### Interpretación

En la fotografía educativa de 2021 existen aproximadamente **2,9 matrículas vinculadas a motores productivos por cada matrícula vinculada a capacidades transformadoras**.

Esto **no demuestra un déficit de talento tecnológico**. Lo que muestra es que, dentro de nuestra clasificación, la base formativa asociada a la estructura productiva era bastante mayor que la asociada a digitalización, automatización y sustentabilidad.


## 4.3 ¿Qué familias forman cada lado?


In [ ]:
#Frecuencia por familia estratégica
cuenta_adn = d[d['ADN_BIOBIO'] != 'Otros campos'].groupby('ADN_BIOBIO').size()

tabla_adn = pd.DataFrame({
    'Frecuencia absoluta': cuenta_adn,
    'Frecuencia relativa (%)': cuenta_adn/n * 100
})

tabla_adn = tabla_adn.sort_values('Frecuencia absoluta', ascending=False)

round(tabla_adn, 2)


In [ ]:
#Variables
categorias_familia = tabla_adn.index
h_familia = tabla_adn['Frecuencia relativa (%)']

#Gráfico
fig, ax = plt.subplots(figsize=(10,6))
barras_familia = ax.bar(categorias_familia, h_familia, edgecolor='black')

#Personalización
ax.set_title('Familias estratégicas del ADN profesional del Biobío')
ax.set_ylabel('Porcentaje de matrículas')
ax.bar_label(barras_familia, fmt='%.1f%%')
plt.xticks(rotation=45, ha='right')

plt.show()


**Interpretación:** Industria y manufactura es la familia estratégica más numerosa, con aproximadamente **11,10%** de toda la matrícula. En el bloque transformador, Digital y TIC representa **3,33%**, Automatización y robótica **2,58%** y Energía/sustentabilidad/biotecnología **1,02%**.


## 4.4 ¿Por qué podría existir esta diferencia y por qué importa?

La base no permite demostrar las causas. Sin embargo, existe una explicación contextual razonable:

- El Biobío posee una larga tradición industrial y productiva, por lo que es esperable encontrar una oferta formativa importante asociada a esos sectores.
- La transformación hacia manufactura avanzada, automatización, tecnologías digitales e Industria 4.0 corresponde a una etapa de modernización que hoy tiene mayor importancia regional.

Si en el futuro la demanda por competencias digitales y automatizadas creciera más rápido que la formación disponible, podrían aparecer dificultades para adoptar tecnología o una mayor dependencia de talento externo. **Estas son consecuencias posibles, no efectos demostrados por el Excel.**


# 5. ÍTEM 2 — Preguntas obligatorias


## Pregunta 1
### ¿Hay áreas del conocimiento donde las carreras sean más caras? ¿Qué criterio diseñaron?

**Criterio:** usamos las **ofertas académicas únicas** y comparamos la **mediana del arancel** de cada área. Elegimos la mediana porque se ve menos afectada por valores extremos que la media.


In [ ]:
#Mediana global de todas las ofertas
mediana_global = ofertas['VALOR ARANCEL (PESOS)'].median()

#Agrupamos las ofertas por área del conocimiento
tabla_p1 = ofertas.groupby('AREA CONOCIMIENTO')['VALOR ARANCEL (PESOS)'].agg(
    ['count', 'median']
)

#Renombramos las columnas
tabla_p1.columns = ['Cantidad de ofertas', 'Arancel mediano']

#Ordenamos de mayor a menor mediana
tabla_p1 = tabla_p1.sort_values('Arancel mediano', ascending=False)

print(f'Mediana global: ${mediana_global:,.0f}')
round(tabla_p1, 0)


In [ ]:
#Variables
categorias_p1 = tabla_p1.index
medianas_p1 = tabla_p1['Arancel mediano']/1000000

#Gráfico
fig, ax = plt.subplots(figsize=(10,5))
barras_p1 = ax.bar(categorias_p1, medianas_p1, edgecolor='black')

#Personalización
ax.set_title('Arancel mediano por área del conocimiento')
ax.set_xlabel('Área del conocimiento')
ax.set_ylabel('Millones de pesos')
plt.xticks(rotation=45, ha='right')

plt.show()


### Respuesta

Sí, existen diferencias entre áreas. La mediana global de las 1.273 ofertas es **$2.030.000**.

Las mayores medianas corresponden a:

1. **Derecho:** $3.586.000.
2. **Ciencias Básicas:** $3.290.000.
3. **Agropecuaria:** $2.981.500.

Por lo tanto, estas áreas presentan aranceles medianos superiores al valor global de las ofertas analizadas.


## Pregunta 2
### ¿Qué influencia tiene la edad en el tipo de institución a la que ingresan los estudiantes?

Como el estudio es descriptivo, hablaremos de **diferencias observadas** entre los grupos de edad y no de causalidad.


In [ ]:
#Tabla de contingencia con frecuencias absolutas
tabla_edad_institucion = pd.crosstab(
    d['RANGO EDAD'],
    d['TIPO DE INSTITUCION']
)

tabla_edad_institucion


In [ ]:
#Transformamos la tabla a porcentajes por fila
tabla_edad_inst_pct = pd.crosstab(
    d['RANGO EDAD'],
    d['TIPO DE INSTITUCION'],
    normalize='index'
) * 100

#Orden lógico de los rangos de edad
orden_edades = [
    '15 a 19 ',
    '20 a 24 ',
    '25 a 29 ',
    '30 a 34 ',
    '35 a 39 ',
    '40 y mas'
]

tabla_edad_inst_pct = tabla_edad_inst_pct.reindex(orden_edades)

round(tabla_edad_inst_pct, 2)


In [ ]:
#Gráfico de barras apiladas al 100%
fig, ax = plt.subplots(figsize=(10,6))

tabla_edad_inst_pct.plot(
    kind='bar',
    stacked=True,
    ax=ax,
    edgecolor='black'
)

#Personalización
ax.set_title('Tipo de institución según rango de edad')
ax.set_xlabel('Rango de edad')
ax.set_ylabel('Porcentaje dentro del rango de edad')
plt.xticks(rotation=0)
plt.legend(title='Tipo de institución', bbox_to_anchor=(1.05,1), loc='upper left')

plt.show()


### Respuesta

Se observan diferencias claras en la composición institucional según la edad.

- Entre **15 y 19 años**, las universidades CRUCH representan aproximadamente **46,24%**, mientras los IP representan **19,30%**.
- Entre quienes tienen **40 años o más**, los IP aumentan a aproximadamente **52,85%** y las universidades CRUCH disminuyen a **10,90%**.

Por lo tanto, la distribución del tipo de institución cambia a medida que cambia el rango de edad. Esto describe una asociación, pero no demuestra que la edad sea la causa de la elección.


## Pregunta 3
### ¿Hay carreras cuyo arancel sea sustantivamente más caro que la mayoría?

Utilizamos el rango intercuartílico:

\[
RIC = Q_3 - Q_1
\]

\[
Límite\ superior = Q_3 + 1,5 	imes RIC
\]

Una oferta se considera sustantivamente alta cuando su arancel supera ese límite.


In [ ]:
#Primer y tercer cuartil
Q1 = ofertas['VALOR ARANCEL (PESOS)'].quantile(0.25)
Q3 = ofertas['VALOR ARANCEL (PESOS)'].quantile(0.75)

#Rango intercuartílico
RIC = Q3 - Q1

#Límite superior
limite = Q3 + 1.5 * RIC

print(f'Q1: ${Q1:,.0f}')
print(f'Q3: ${Q3:,.0f}')
print(f'RIC: ${RIC:,.0f}')
print(f'Límite superior: ${limite:,.0f}')


In [ ]:
#Filtramos las ofertas que superan el límite
altas = ofertas[
    ofertas['VALOR ARANCEL (PESOS)'] > limite
].copy()

#Cantidad y porcentaje
cantidad_altas = altas.shape[0]
porcentaje_altas = cantidad_altas/ofertas.shape[0] * 100

print(f'Ofertas sobre el límite: {cantidad_altas}')
print(f'Porcentaje: {porcentaje_altas:.1f}%')


In [ ]:
#Frecuencia de ofertas altas por tipo de institución
cuenta_altas_inst = altas.groupby('TIPO DE INSTITUCION').size()

tabla_altas_inst = pd.DataFrame({
    'Ofertas con arancel alto': cuenta_altas_inst
})

tabla_altas_inst.sort_values(
    'Ofertas con arancel alto',
    ascending=False
)


In [ ]:
#Carreras con mayor arancel mediano
ranking_carreras = ofertas.groupby('NOMBRE CARRERA')['VALOR ARANCEL (PESOS)'].agg(
    ['count', 'median']
)

ranking_carreras.columns = ['Cantidad de ofertas', 'Arancel mediano']

#Consideramos carreras con al menos 3 ofertas
ranking_carreras = ranking_carreras[
    ranking_carreras['Cantidad de ofertas'] >= 3
]

ranking_carreras = ranking_carreras.sort_values(
    'Arancel mediano',
    ascending=False
).head(10)

ranking_carreras


### Respuesta

Sí. El límite superior obtenido mediante el criterio del RIC es **$4.026.000**.

Superan ese valor **135 de las 1.273 ofertas**, equivalentes a aproximadamente **10,6%**.

Las 135 ofertas pertenecen a universidades: **73 a universidades CRUCH y 62 a universidades privadas**.

Entre las carreras con mayor arancel mediano aparecen Medicina y Odontología. La base permite identificar estas diferencias, pero no permite afirmar por sí sola la causa de esos precios.


# 6. Conclusiones generales

### Conclusión estadística principal
La fotografía de 2021 muestra que, según nuestra clasificación:

- **20,14%** de las matrículas está relacionado con motores productivos.
- **6,93%** está relacionado con capacidades transformadoras.

Esto equivale aproximadamente a **2,9 matrículas productivas por cada matrícula transformadora**.

### Relación con el Biobío
El resultado permite describir el punto de partida formativo de una región que posee una fuerte tradición industrial y que actualmente busca fortalecer manufactura avanzada e Industria 4.0.

### Conclusión del trabajo
**En 2021, la formación vinculada a los motores productivos del Biobío era considerablemente mayor que la formación clasificada como digital, automatizada y sustentable.**

Esto no significa que exista necesariamente un déficit. Para comprobarlo sería necesario incorporar información de empleo, vacantes, salarios y demanda de competencias.


# 7. Limitaciones

- La base corresponde a 2021.
- La clasificación “Acero/Algoritmo” fue creada para este trabajo a partir del nombre de las carreras.
- Una carrera puede aportar a más de un sector, aunque aquí se asigna a una sola familia para evitar doble conteo.
- La base no contiene demanda laboral, vacantes, salarios ni productividad.
- Los aranceles corresponden a valores registrados en la base y no necesariamente al monto efectivo pagado por cada estudiante.
- Los resultados son descriptivos y no demuestran causalidad.
